# Chapter 6: Approximation and Fitting

## Overview
In many fields, we face the problem of finding a model or a mathematical function that closely approximates a set of given data points or an underlying true function. Convex optimization provides powerful, systematic tools for solving these **approximation and fitting** problems.

### Key Concepts
1. **Norm Approximation:** The core idea is to find a parameter vector $x$ that minimizes the residual $r = Ax - b$. We measure the "size" of the residual using different norms:
   - **$L_2$ Norm (Least Squares):** Minimizes the sum of squared errors. It is statistically optimal for Gaussian noise but highly sensitive to outliers.
   - **$L_1$ Norm (Robust Approximation):** Minimizes the sum of absolute errors. It is heavily used when the data contains outliers because it doesn't penalize large errors quadratically.
   - **$L_\infty$ Norm (Chebyshev/Minimax Approximation):** Minimizes the maximum absolute error. Used when the worst-case error must be bounded (e.g., in aerospace control systems).

2. **Least-Norm Problems:** When a system of equations $Ax = b$ has infinitely many solutions (underdetermined), we often want the "simplest" or "smallest" solution by minimizing $\|x\|$. 
   - Minimizing $\|x\|_2$ spreads the values out.
   - Minimizing $\|x\|_1$ promotes **sparsity** (solutions where many entries are exactly zero).

3. **Regularized Approximation:** Instead of just minimizing the error $\|Ax - b\|$, we minimize a combination of the error and a penalty on the size of $x$:

$$
\text{minimize} \quad \|Ax - b\|_2^2 + \gamma \|x\|
$$

   - If we penalize $\|x\|_2^2$, this is **Tikhonov Regularization (Ridge Regression)**, which prevents overfitting.
   - If we penalize $\|x\|_1$, this is **Lasso**, which automatically performs feature selection by driving some coefficients to zero.

## Applications & Problems Solved
- **Machine Learning & Statistics:** Linear regression, robust regression, and feature selection (Lasso).
- **Signal Processing:** Signal restoration and compressed sensing (recovering sparse signals from few measurements).
- **Control Engineering:** System identification (finding a mathematical model that explains observed system behavior).

## Code Example
See `norm_approximation.py` for a demonstration of how different norms behave when fitting a line to data with **outliers**. We compare standard Least Squares ($L_2$) versus Robust Approximation ($L_1$).

![Norm Approximation](approximation.png)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp

def demonstrate_norm_approximation():
    """
    Demonstrates the difference between L2 (Least Squares) and 
    L1 (Robust) norm approximation in the presence of outliers.
    """
    # 1. Generate synthetic data (a straight line with noise)
    np.random.seed(42)
    m = 50  # Number of data points
    n = 2   # Number of features (slope and intercept)
    
    # x values from 0 to 10
    x_data = np.linspace(0, 10, m)
    
    # True parameters: slope = 2, intercept = 1
    true_theta = np.array([2.0, 1.0])
    
    # Construct A matrix for A * theta = b
    A = np.vstack([x_data, np.ones(m)]).T
    
    # Generate b with small Gaussian noise
    b = A @ true_theta + np.random.randn(m) * 0.5
    
    # Add severe outliers to the data
    b[40] -= 15
    b[42] -= 20
    b[45] += 18
    b[48] += 25

    # 2. L2 Norm Approximation (Least Squares)
    theta_l2 = cp.Variable(n)
    cost_l2 = cp.sum_squares(A @ theta_l2 - b)
    prob_l2 = cp.Problem(cp.Minimize(cost_l2))
    prob_l2.solve()
    
    # 3. L1 Norm Approximation (Robust Approximation)
    theta_l1 = cp.Variable(n)
    cost_l1 = cp.norm1(A @ theta_l1 - b)
    prob_l1 = cp.Problem(cp.Minimize(cost_l1))
    prob_l1.solve()

    # 4. Plot the results
    plt.figure(figsize=(10, 6))
    plt.plot(x_data, b, 'ko', label='Data with outliers', markersize=5)
    
    # True line
    plt.plot(x_data, A @ true_theta, 'g-', lw=2, label='True Line')
    
    # L2 line
    plt.plot(x_data, A @ theta_l2.value, 'r--', lw=2, label='L2 Approximation (Least Squares)')
    
    # L1 line
    plt.plot(x_data, A @ theta_l1.value, 'b-.', lw=2, label='L1 Approximation (Robust)')
    
    plt.title("Norm Approximation: L1 vs L2 in the presence of outliers")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig("approximation.png")
    
    print("Approximation complete. Plot saved as approximation.png")

if __name__ == "__main__":
    demonstrate_norm_approximation()
